# Ablations of SiameseFit and tests of the max-pooling

The trained encoder of SiameseFit is replaced by fixed operators, while the normalization of the data, the
form of the loss and the optimizer of the velocity model (the calibrated settings of SiameseFit) are kept.
The ablations start from the linear initial model.

**Keys**
- `model`: `'over'` or `'marm'`
- `variant`, ablation ladder (Sections IV-A2 and IV-B2):
  - `'identity'`: no encoder
  - `'avgpool'`: 16 x 16 average pooling with stride 16
  - `'randproj'`: fixed random linear projection with the same downsampling and 128 channels
  - `'maxpool'`: 16 x 16 max-pooling with stride 16
  - `'skip128'`: the last skip branch of the frozen SiameseFit network alone, with its 128 random 3 x 3 filters,
    LeakyReLU and 16 x 16 max-pooling
  - `'frozen'`: SiameseFit network kept at its random initialization
  - `'maxpool_l2'`: 16 x 16 max-pooling compared with the L2 loss of MPBAE instead of the SiameseFit loss
- `variant`, tests of the max-pooling on the Marmousi2 model (Section IV-B2):
  - `'maxpool_shift'`: grid of the blocks shifted at random at each update
  - `'maxpool_top16'`: mean of the 16 largest samples of each block
  - `'maxpool_abs'`: max-pooling of the absolute value of the data
  - `'maxpool_mute'`: early arrivals muted in both gathers
- `seed`: each variant is run with three seeds in the article (Figs 4 and 11, `run_all.sh`)

Outputs are written to `results/<model>_linear/<variant>/seed<seed>/`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from utils.fwi import run, run_dir, calibrated
from utils.plots import NAMES, plot_models, plot_curves

In [ ]:
model = 'over'      # 'over' or 'marm'
variant = 'skip128'  # 'identity', 'avgpool', 'randproj', 'maxpool', 'skip128', 'frozen',
                    # 'maxpool_l2', 'maxpool_shift', 'maxpool_top16', 'maxpool_abs' or 'maxpool_mute'
seed = 0

init, method = 'linear', variant
lr, eps = calibrated(model, init, method)
print('Step of the velocity model: %g m/s, Adam eps: %g' % (lr, eps))


In [ ]:
metrics = run(model, init, method, seed)

In [ ]:
inverted = np.load(run_dir(model, init, method, seed) + '/model_final.npy')
plot_models(model, init, inverted, NAMES[method])
plt.show()
plot_curves(metrics, NAMES[method])
plt.show()
print('Final SNR %.2f dB, SSIM %.3f' % (metrics['snr'][-1], metrics['ssim'][-1]))